Quickstart: Compare runs, choose a model, and deploy it to a REST API

In this quickstart, you will:

• Run a hyperparameter sweep on a training script  
• Compare the results of the runs in the MLflow UI  
• Choose the best run and register it as a model  
• Deploy the model to a REST API  
• Build a container image suitable for deployment to a cloud platform

In [25]:
import keras
import numpy as np
import pandas as pd
from hyperopt import STATUS_OK,Trials,fmin,hp,tpe
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.models import infer_signature

In [26]:
## load the dataset
data=pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",
)
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [27]:
## Split the data into training,validation and test sets

train,test=train_test_split(data,test_size=0.25,random_state=42)
train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
2835,6.3,0.25,0.22,3.30,0.048,41.0,161.0,0.99256,3.16,0.50,10.5,6
1157,7.8,0.30,0.29,16.85,0.054,23.0,135.0,0.99980,3.16,0.38,9.0,6
744,7.4,0.38,0.27,7.50,0.041,24.0,160.0,0.99535,3.17,0.43,10.0,5
1448,7.4,0.16,0.49,1.20,0.055,18.0,150.0,0.99170,3.23,0.47,11.2,6
3338,7.2,0.27,0.28,15.20,0.046,6.0,41.0,0.99665,3.17,0.39,10.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4426,6.2,0.21,0.52,6.50,0.047,28.0,123.0,0.99418,3.22,0.49,9.9,6
466,7.0,0.14,0.32,9.00,0.039,54.0,141.0,0.99560,3.22,0.43,9.4,6
3092,7.6,0.27,0.52,3.20,0.043,28.0,152.0,0.99129,3.02,0.53,11.4,6
3772,6.3,0.24,0.29,13.70,0.035,53.0,134.0,0.99567,3.17,0.38,10.6,6


In [28]:
train[['quality']].values.ravel()

array([6, 6, 5, ..., 6, 6, 8], shape=(3673,))

In [29]:
train_x=train.drop(['quality'],axis=1).values
train_y=train[['quality']].values.ravel()

## test dataset
test_x=test.drop(['quality'],axis=1).values
test_y=test[['quality']].values.ravel()

## splitting this train data into train and validation

train_x,valid_x,train_y,valid_y=train_test_split(train_x,train_y,test_size=0.20,random_state=42)

signature=infer_signature(train_x,train_y)

In [30]:
### ANN Model

def train_model(params,epochs,train_x,train_y,valid_x,valid_y,test_x,test_y):

    ## Define model architecture
    mean=np.mean(train_x,axis=0)
    var=np.var(train_x,axis=0)

    model=keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean,variance=var),
            keras.layers.Dense(64,activation='relu'),
            keras.layers.Dense(1)

        ]
    )

    ## compile the model
    model.compile(optimizer=keras.optimizers.SGD(
        learning_rate=params["lr"],momentum=params["momentum"]
    ),
    loss="mean_squared_error",
    metrics=[keras.metrics.RootMeanSquaredError()]
    )

    ## Train the ANN model with lr and momentum params wwith MLFLOW tracking
    with mlflow.start_run(nested=True):
        model.fit(train_x,train_y,validation_data=(valid_x,valid_y),
                  epochs=epochs,
                  batch_size=64)
        
        ## Evaluate the model
        eval_result=model.evaluate(valid_x,valid_y,batch_size=64)

        eval_rmse=eval_result[1]

        ## Log the parameters and results
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse",eval_rmse)

        ## log the model

        mlflow.tensorflow.log_model(model,"model",signature=signature)

        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

In [31]:
def objective(params):
    # MLflow will track the parameters and results for each run
    result = train_model(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y,
    )
    return result

In [32]:
space={
    "lr":hp.loguniform("lr",np.log(1e-5),np.log(1e-1)),
    "momentum":hp.uniform("momentum",0.0,1.0)

}

In [33]:
mlflow.set_experiment("wine-quality")
with mlflow.start_run():
    # Conduct the hyperparameter search using Hyperopt
    trials=Trials()
    best=fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials
    )

    # Fetch the details of the best run
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

    # Print out the best parameters and corresponding loss
    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")


Epoch 1/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 16s 374ms/step - loss: 37.6760 - root_mean_squared_error: 6.1381
35/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 37.3233 - root_mean_squared_error: 6.1091   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 36.4121 - root_mean_squared_error: 6.0342 - val_loss: 35.8142 - val_root_mean_squared_error: 5.9845

Epoch 2/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 36.2936 - root_mean_squared_error: 6.0244
33/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 35.6942 - root_mean_squared_error: 5.9742 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 34.5587 - root_mean_squared_error: 5.8787 - val_loss: 33.9987 - val_root_mean_squared_error: 5.8308

Epoch 3/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - loss: 33.3509 - root_mean_squared_error: 5.7750
32/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 33.4881 - root_mean_squared_error: 5.7

2026/03/11 19:52:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/3                                                                    

 1/46 ━━━━━━━━━━━━━━━━━━━━ 16s 370ms/step - loss: 37.2126 - root_mean_squared_error: 6.1002
31/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 10.7381 - root_mean_squared_error: 3.1072   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 3.3560 - root_mean_squared_error: 1.8319 - val_loss: 1.0143 - val_root_mean_squared_error: 1.0071

Epoch 2/3                                                                    

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.9268 - root_mean_squared_error: 0.9627
35/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.8901 - root_mean_squared_error: 0.9431 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.7887 - root_mean_squared_error: 0.8881 - val_loss: 0.7217 - val_root_mean_squared_error: 0.8496

Epoch 3/3                                                                    

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.6197 - root_mean_squared_error: 0.7872
36/46 ━━━━━━━━━━━━━━

2026/03/11 19:52:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 15s 350ms/step - loss: 36.4562 - root_mean_squared_error: 6.0379
32/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 15.2512 - root_mean_squared_error: 3.7666   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 5.2568 - root_mean_squared_error: 2.2928 - val_loss: 1.2693 - val_root_mean_squared_error: 1.1266

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 2.0567 - root_mean_squared_error: 1.4341
36/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.1712 - root_mean_squared_error: 1.0792 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.9958 - root_mean_squared_error: 0.9979 - val_loss: 0.8611 - val_root_mean_squared_error: 0.9280

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.6835 - root_mean_squared_error: 0.8267
37/46 ━━━━━━━━

...

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 15s 347ms/step - loss: 36.0204 - root_mean_squared_error: 6.0017
33/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 7.9506 - root_mean_squared_error: 2.6386    
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 2.5880 - root_mean_squared_error: 1.6087 - val_loss: 0.9019 - val_root_mean_squared_error: 0.9497

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.7524 - root_mean_squared_error: 0.8674
37/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.7431 - root_mean_squared_error: 0.8620 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.7248 - root_mean_squared_error: 0.8514 - val_loss: 0.6329 - val_root_mean_squared_error: 0.7956

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.7961 - root_mean_squared_error: 0.8922
41/46 ━━━━━━━━

2026/03/11 19:52:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



100%|██████████| 4/4 [00:47<00:00, 11.76s/trial, best loss: 0.7660351395606995]

2026/03/11 19:53:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best parameters: {'lr': np.float64(0.017328864026180564), 'momentum': np.float64(0.5853404330287749)}
Best eval rmse: 0.7660351395606995
